In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
TRAIN_DIR = r"C:\Users\Acer\Downloads\dldata\Backup_sampled\train"
VAL_DIR = r"C:\Users\Acer\Downloads\dldata\Backup_sampled\test"
TEST_DIR = r"C:\Users\Acer\Downloads\dldata\Backup_sampled\val"


In [4]:
transform = transforms.Compose([
    transforms.Resize((224,224)), 
    transforms.ToTensor(),                       
    transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)




print(f"Total Training images: {len(train_dataset)}")
print(f"Total Validation images: {len(val_dataset)}")
print(f"Total Test images: {len(test_dataset)}")
print("Class mapping:", train_dataset.class_to_idx)

Total Training images: 20249
Total Validation images: 500
Total Test images: 500
Class mapping: {'fake': 0, 'real': 1}


In [21]:
class VGG16(nn.Module):
    def __init__(self, num_classes=2):
        super(VGG16, self).__init__()

        self.features = nn.Sequential(
            # Block 1 
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2 (2 conv layers)
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 3 
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 4 
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 5 (3 conv layers)
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        # Classifier layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.LazyLinear(num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [22]:
model = VGG16(num_classes=2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [23]:
losses = []
test_loss = []
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch, y in train_loader:
        batch = batch.to(device)
        y = y.to(device)

        optimizer.zero_grad()              
        prediction = model(batch)
        loss = criterion(prediction, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        losses.append(loss.item())

    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch} | Train loss: {avg_epoch_loss:.4f}")

    #eval
    model.eval()
    test_epoch_loss = 0.0

    with torch.no_grad():
        for batch, y in test_loader:
            batch = batch.to(device)
            y = y.to(device)

            prediction = model(batch)
            loss = criterion(prediction, y)
            test_epoch_loss += loss.item()
            test_loss.append(loss.item())

    avg_test_loss = test_epoch_loss / len(test_loader)
    print(f"Epoch {epoch} | Test loss: {avg_test_loss:.4f}")
    
def accuracy(model, data_loader):
    correct = 0
    total = 0
    model.eval()

    with torch.no_grad():
        for X, y in data_loader:
            X = X.to(device)
            y = y.to(device)

            preds = model(X)
            _, predicted = torch.max(preds, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    return 100 * correct / total
print("Accuracy on test set:", accuracy(model, test_loader), "%")


Epoch 0 | Train loss: 0.8956
Epoch 0 | Test loss: 0.6152
Epoch 1 | Train loss: 0.5933
Epoch 1 | Test loss: 0.5589
Epoch 2 | Train loss: 0.5341
Epoch 2 | Test loss: 0.5417
Epoch 3 | Train loss: 0.4883
Epoch 3 | Test loss: 0.4368
Epoch 4 | Train loss: 0.4411
Epoch 4 | Test loss: 0.4708
Epoch 5 | Train loss: 0.3781
Epoch 5 | Test loss: 0.3201
Epoch 6 | Train loss: 0.3368
Epoch 6 | Test loss: 0.3365
Epoch 7 | Train loss: 0.2993
Epoch 7 | Test loss: 0.3785
Epoch 8 | Train loss: 0.2770
Epoch 8 | Test loss: 0.2749
Epoch 9 | Train loss: 0.2574
Epoch 9 | Test loss: 0.2901
Accuracy on test set: 87.0 %


In [24]:
torch.save(model.state_dict(), "vgg16_binary.pth")

In [14]:
class AlexNetInspo(nn.Module):
    def __init__(self, num_classes =2):
        super(AlexNetInspo, self).__init__()

        self.features = nn.Sequential(
            #first layer
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                
            #second
            nn.Conv2d(64,192, kernel_size=5, padding=2),         
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),               
            #last one
            nn.Conv2d(192,384, kernel_size=3, padding=1),        
            nn.ReLU(inplace=True),
            nn.Conv2d(384,256, kernel_size=3, padding=1),        
            nn.ReLU(inplace=True),
            nn.Conv2d(256,256, kernel_size=3, padding=1),        
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                
        )
        # Classifier layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.LazyLinear(4096),
            nn.ReLU(),
            nn.LazyLinear(num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
model2 = AlexNetInspo(num_classes=2)
model2 = model2.to(device)
print(model2)

AlexNetInspo(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): LazyLinear(in_features=0, out_features=4096, bias=True)
    (2): ReLU()
    (3): LazyLinear(in_feat

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)

In [19]:
losses = []
test_loss = []
num_epochs = 10

for epoch in range(num_epochs):
    model2.train()
    epoch_loss = 0.0

    for batch, y in train_loader:
        batch = batch.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        prediction = model2(batch)
        loss = criterion(prediction, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch} | Train loss: {avg_epoch_loss:.4f}")

   
    # Evaluate on test set
    model2.eval()
    test_epoch_loss = 0.0

    with torch.no_grad():
     for batch, y in test_loader:
        batch = batch.to(device)
        y = y.to(device)

        prediction = model2(batch)
        loss = criterion(prediction, y)
        test_epoch_loss += loss.item()

    avg_test_loss = test_epoch_loss / len(test_loader)
    print(f"Epoch {epoch} | Test loss: {avg_test_loss:.4f}")


def accuracy(model2, data_loader):
    correct = 0
    total = 0
    model2.eval()
    with torch.no_grad():
        for X, y in data_loader:
            X = X.to(device)
            y = y.to(device)
           
            preds = model2(X)
            _, predicted = torch.max(preds, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
    return 100 * correct / total

print("Accuracy on test set:", accuracy(model2, test_loader), "%")

Epoch 0 | Train loss: 0.6789
Epoch 0 | Test loss: 0.6732
Epoch 1 | Train loss: 0.6291
Epoch 1 | Test loss: 0.5825
Epoch 2 | Train loss: 0.5363
Epoch 2 | Test loss: 0.5423
Epoch 3 | Train loss: 0.4615
Epoch 3 | Test loss: 0.4450
Epoch 4 | Train loss: 0.4193
Epoch 4 | Test loss: 0.3996
Epoch 5 | Train loss: 0.3900
Epoch 5 | Test loss: 0.4352
Epoch 6 | Train loss: 0.3617
Epoch 6 | Test loss: 0.4138
Epoch 7 | Train loss: 0.3299
Epoch 7 | Test loss: 0.3811
Epoch 8 | Train loss: 0.3044
Epoch 8 | Test loss: 0.4030
Epoch 9 | Train loss: 0.2796
Epoch 9 | Test loss: 0.4507
Accuracy on test set: 81.8 %


In [20]:
torch.save(model2.state_dict(), "alex_binary.pth")
